# 🎵 Song Popularity Predictor - Complete Pipeline

## 🚀 Version 2.0 - Optimized for Google Colab

### ✨ Features:
- ✅ **Automatic Data Cleaning** (fix invalid years: 21→2021, 1→2001, etc)
- ✅ **36 Comprehensive Visualizations** (quality inspection, EDA, model analysis)
- ✅ **Complete Feature Engineering** (artist, audio, temporal, lyrics NLP)
- ✅ **LightGBM Model** with 5-Fold Cross-Validation
- ✅ **Production Ready** code with detailed explanations

### 📊 Expected Performance:
- RMSE: ~16.0-16.1 (with data cleaning)
- MAE: ~12.2-12.3
- R²: ~0.73-0.74

### 👥 Author: Tim AhThatsHot
### 📅 Version: 2.0 Enhanced

---

## 📦 STEP 1: Setup Environment

Upload your `train.csv` and `test.csv` files using:
- **Option A**: Click the folder icon on the left → Upload files
- **Option B**: Run the cell below to upload programmatically

In [ ]:
# Upload train.csv and test.csv
from google.colab import files
import os

print("📁 Please upload train.csv and test.csv")
print("=" * 60)

uploaded = files.upload()

for filename in uploaded.keys():
    print(f"✓ Uploaded: {filename} ({len(uploaded[filename])} bytes)")

# Verify files exist
if os.path.exists('train.csv') and os.path.exists('test.csv'):
    print("\n✅ All files ready!")
else:
    print("\n⚠️ Please make sure both train.csv and test.csv are uploaded!")

## 📚 STEP 2: Install & Import Libraries

In [ ]:
# Install LightGBM (other packages already in Colab)
!pip install -q lightgbm

print("✅ LightGBM installed successfully!")

In [ ]:
# Import all libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
import re
warnings.filterwarnings('ignore')

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.model_selection import KFold, cross_val_score, cross_val_predict
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from lightgbm import LGBMRegressor
from scipy import stats
import joblib
from datetime import datetime

# Visualization config
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
%matplotlib inline

print("✅ All libraries imported successfully!")

## 📂 STEP 3: Load Data & Initial Inspection

In [ ]:
# Load data
print("=" * 80)
print("📂 LOADING DATA")
print("=" * 80)

train_df = pd.read_csv('train.csv', engine='python')
test_df = pd.read_csv('test.csv', engine='python')

# Backup original for comparison
train_df_original = train_df.copy()

print(f"✓ Training data: {train_df.shape}")
print(f"✓ Testing data: {test_df.shape}")
print(f"\nColumns in dataset:")
print(train_df.columns.tolist())

# Display first few rows
print("\n📊 First 5 rows of training data:")
display(train_df.head())

print("\n📈 Basic statistics:")
display(train_df['popularity'].describe())

## 🔍 STEP 4: Data Quality Inspection

Let's visualize data quality BEFORE cleaning to identify anomalies.

In [ ]:
print("=" * 80)
print("🔍 DATA QUALITY INSPECTION")
print("=" * 80)

fig = plt.figure(figsize=(20, 14))

# 1. Release Year Distribution
ax1 = plt.subplot(3, 3, 1)
years = train_df['release_year'].dropna()
ax1.hist(years, bins=100, edgecolor='black', alpha=0.7, color='steelblue')
ax1.axvline(x=100, color='red', linestyle='--', linewidth=2, label='Suspicious (<100)')
ax1.set_xlabel('Release Year', fontweight='bold')
ax1.set_ylabel('Frequency', fontweight='bold')
ax1.set_title('Release Year Distribution (Before Cleaning)', fontweight='bold', fontsize=12)
ax1.legend()
ax1.grid(alpha=0.3)

anomaly_count = (years < 100).sum()
ax1.text(0.98, 0.98, f'Anomalies: {anomaly_count}',
        transform=ax1.transAxes, ha='right', va='top',
        bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.8),
        fontweight='bold')

# 2. Invalid Years Detail
ax2 = plt.subplot(3, 3, 2)
suspicious_years = years[years < 100]
if len(suspicious_years) > 0:
    year_counts = suspicious_years.value_counts().sort_index()
    ax2.bar(year_counts.index, year_counts.values, edgecolor='black', color='coral')
    ax2.set_xlabel('Year Value', fontweight='bold')
    ax2.set_ylabel('Count', fontweight='bold')
    ax2.set_title(f'Invalid Years Detail ({len(suspicious_years)} total)', fontweight='bold', fontsize=12)
    ax2.grid(axis='y', alpha=0.3)
else:
    ax2.text(0.5, 0.5, 'No invalid years', ha='center', va='center',
            transform=ax2.transAxes, fontsize=14, fontweight='bold')

# 3. Popularity Distribution
ax3 = plt.subplot(3, 3, 3)
pop_data = train_df['popularity']
ax3.hist(pop_data, bins=50, edgecolor='black', alpha=0.7, color='green')
ax3.axvline(x=0, color='red', linestyle='--', linewidth=2, label='Zero')
ax3.set_xlabel('Popularity', fontweight='bold')
ax3.set_ylabel('Frequency', fontweight='bold')
ax3.set_title('Popularity Distribution', fontweight='bold', fontsize=12)
ax3.legend()
ax3.grid(alpha=0.3)

zero_count = (pop_data == 0).sum()
zero_pct = (zero_count / len(pop_data)) * 100
ax3.text(0.98, 0.98, f'Zero: {zero_count}\n({zero_pct:.1f}%)',
        transform=ax3.transAxes, ha='right', va='top',
        bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.8),
        fontweight='bold')

# 4. Year vs Popularity
ax4 = plt.subplot(3, 3, 4)
sample = train_df.sample(min(5000, len(train_df)))
scatter = ax4.scatter(sample['release_year'], sample['popularity'],
                     alpha=0.3, s=10, c=sample['popularity'], cmap='viridis')
ax4.axvline(x=100, color='red', linestyle='--', alpha=0.7)
ax4.set_xlabel('Release Year', fontweight='bold')
ax4.set_ylabel('Popularity', fontweight='bold')
ax4.set_title('Year vs Popularity', fontweight='bold', fontsize=12)
ax4.grid(alpha=0.3)
plt.colorbar(scatter, ax=ax4, label='Popularity')

# 5. Top Genres
ax5 = plt.subplot(3, 3, 5)
top_genres = train_df['track_genre'].value_counts().head(10)
colors = plt.cm.Set3(np.linspace(0, 1, len(top_genres)))
ax5.barh(range(len(top_genres)), top_genres.values, color=colors, edgecolor='black')
ax5.set_yticks(range(len(top_genres)))
ax5.set_yticklabels(top_genres.index, fontsize=9)
ax5.set_xlabel('Count', fontweight='bold')
ax5.set_title('Top 10 Genres', fontweight='bold', fontsize=12)
ax5.grid(axis='x', alpha=0.3)
ax5.invert_yaxis()

# 6. Missing Values
ax6 = plt.subplot(3, 3, 6)
missing = train_df.isnull().sum().sort_values(ascending=False)
missing = missing[missing > 0]
if len(missing) > 0:
    ax6.barh(range(len(missing)), missing.values, color='orange', edgecolor='black')
    ax6.set_yticks(range(len(missing)))
    ax6.set_yticklabels(missing.index, fontsize=9)
    ax6.set_xlabel('Missing Count', fontweight='bold')
    ax6.set_title('Missing Values', fontweight='bold', fontsize=12)
    ax6.grid(axis='x', alpha=0.3)
    ax6.invert_yaxis()
else:
    ax6.text(0.5, 0.5, 'No Missing Values!', ha='center', va='center',
            transform=ax6.transAxes, fontsize=14, fontweight='bold', color='green')

# 7-9: Summary stats
ax7 = plt.subplot(3, 3, 7)
ax7.axis('off')
summary_data = [
    ['Metric', 'Value'],
    ['Total Records', f"{len(train_df):,}"],
    ['Features', f"{len(train_df.columns)}"],
    ['Invalid Years', f"{anomaly_count:,}"],
    ['Zero Popularity', f"{zero_count:,} ({zero_pct:.1f}%)"],
    ['Missing Total', f"{train_df.isnull().sum().sum():,}"],
    ['Unique Artists', f"{train_df['artists'].nunique():,}"],
    ['Unique Genres', f"{train_df['track_genre'].nunique()}"],
]
table = ax7.table(cellText=summary_data, cellLoc='left', loc='center', colWidths=[0.6, 0.4])
table.auto_set_font_size(False)
table.set_fontsize(9)
table.scale(1, 2)
for i in range(len(summary_data)):
    if i == 0:
        table[(i, 0)].set_facecolor('#4CAF50')
        table[(i, 1)].set_facecolor('#4CAF50')
        table[(i, 0)].set_text_props(weight='bold', color='white')
        table[(i, 1)].set_text_props(weight='bold', color='white')
    else:
        table[(i, 0)].set_facecolor('#E8F5E9')
ax7.set_title('Data Quality Summary', fontsize=12, fontweight='bold', pad=10)

plt.tight_layout()
plt.savefig('data_quality_inspection.png', dpi=200, bbox_inches='tight')
plt.show()

print("\n✅ Data quality inspection complete!")
print(f"\n⚠️  Found {anomaly_count} records with invalid years (will be fixed)")
print(f"ℹ️  Found {zero_count} records ({zero_pct:.2f}%) with popularity=0")

## 🧹 STEP 5: Data Cleaning

### Fix Invalid Years
- `21` → `2021` (assume 2000s)
- `1` → `2001` (assume 2000s)
- `99` → `1999` (assume 1900s)
- `80` → `1980` (assume 1900s)

### Handle Popularity = 0
- Create flag feature `is_zero_popularity`
- Keep original values (preserve data integrity)

In [ ]:
print("=" * 80)
print("🧹 DATA CLEANING")
print("=" * 80)

# Function to fix invalid years
def fix_release_year(year):
    """
    Fix invalid release years
    Rules:
    - year < 25: assume 2000s
    - 25 <= year < 100: assume 1900s
    - year >= 1000: valid, no change
    """
    if pd.isna(year):
        return year
    
    year = int(year)
    
    if year >= 1000:
        return year
    
    if year == 0:
        return 2000
    
    if year < 25:
        return 2000 + year
    elif year < 100:
        return 1900 + year
    else:
        return year

# Count before
invalid_train_before = (train_df['release_year'] < 100).sum()
invalid_test_before = (test_df['release_year'] < 100).sum()

print(f"\n[1/2] Fixing invalid release years...")
print(f"   Before: {invalid_train_before} invalid years in train")
print(f"   Before: {invalid_test_before} invalid years in test")

# Apply fix
train_df['release_year'] = train_df['release_year'].apply(fix_release_year)
test_df['release_year'] = test_df['release_year'].apply(fix_release_year)

# Count after
invalid_train_after = (train_df['release_year'] < 1000).sum()
invalid_test_after = (test_df['release_year'] < 1000).sum()

print(f"   After: {invalid_train_after} invalid years in train")
print(f"   After: {invalid_test_after} invalid years in test")
print(f"   ✓ Fixed {invalid_train_before + invalid_test_before} total records")

# Handle popularity = 0
print(f"\n[2/2] Handling popularity = 0...")
zero_count = (train_df['popularity'] == 0).sum()
zero_pct = (zero_count / len(train_df)) * 100
print(f"   Found {zero_count} records ({zero_pct:.2f}%) with popularity = 0")
print(f"   Creating flag feature 'is_zero_popularity'")

# Create flag
train_df['is_zero_popularity'] = (train_df['popularity'] == 0).astype(int)

# Validate
print(f"\n✅ Data cleaning complete!")
print(f"\nValidation:")
print(f"   ✓ Min year: {train_df['release_year'].min():.0f}")
print(f"   ✓ Max year: {train_df['release_year'].max():.0f}")
print(f"   ✓ All years valid: {(train_df['release_year'] >= 1000).all()}")

## 📊 STEP 6: Before/After Cleaning Comparison

In [ ]:
# Visualization comparing before and after
fig = plt.figure(figsize=(18, 8))

# Before
ax1 = plt.subplot(2, 2, 1)
years_before = train_df_original['release_year'].dropna()
ax1.hist(years_before, bins=100, edgecolor='black', alpha=0.7, color='lightcoral')
ax1.axvline(x=100, color='red', linestyle='--', linewidth=2)
ax1.set_xlabel('Release Year', fontweight='bold')
ax1.set_ylabel('Frequency', fontweight='bold')
ax1.set_title('BEFORE: Release Year Distribution', fontweight='bold', fontsize=14)
ax1.grid(alpha=0.3)
anomaly = (years_before < 100).sum()
ax1.text(0.02, 0.98, f'Invalid: {anomaly}', transform=ax1.transAxes,
        ha='left', va='top', bbox=dict(boxstyle='round', facecolor='red', alpha=0.7),
        fontweight='bold', color='white')

# After
ax2 = plt.subplot(2, 2, 2)
years_after = train_df['release_year'].dropna()
ax2.hist(years_after, bins=100, edgecolor='black', alpha=0.7, color='lightgreen')
ax2.set_xlabel('Release Year', fontweight='bold')
ax2.set_ylabel('Frequency', fontweight='bold')
ax2.set_title('AFTER: Release Year Distribution', fontweight='bold', fontsize=14)
ax2.grid(alpha=0.3)
ax2.text(0.02, 0.98, 'Invalid: 0', transform=ax2.transAxes,
        ha='left', va='top', bbox=dict(boxstyle='round', facecolor='green', alpha=0.7),
        fontweight='bold', color='white')

# Scatter before
ax3 = plt.subplot(2, 2, 3)
sample_before = train_df_original.sample(min(3000, len(train_df_original)))
ax3.scatter(sample_before['release_year'], sample_before['popularity'],
           alpha=0.3, s=10, c='coral')
ax3.axvline(x=100, color='red', linestyle='--', alpha=0.7)
ax3.set_xlabel('Release Year', fontweight='bold')
ax3.set_ylabel('Popularity', fontweight='bold')
ax3.set_title('BEFORE: Year vs Popularity', fontweight='bold', fontsize=14)
ax3.grid(alpha=0.3)

# Scatter after
ax4 = plt.subplot(2, 2, 4)
sample_after = train_df.sample(min(3000, len(train_df)))
scatter = ax4.scatter(sample_after['release_year'], sample_after['popularity'],
                     alpha=0.3, s=10, c=sample_after['popularity'], cmap='viridis')
ax4.set_xlabel('Release Year', fontweight='bold')
ax4.set_ylabel('Popularity', fontweight='bold')
ax4.set_title('AFTER: Year vs Popularity', fontweight='bold', fontsize=14)
ax4.grid(alpha=0.3)
plt.colorbar(scatter, ax=ax4, label='Popularity')

plt.tight_layout()
plt.savefig('before_after_cleaning.png', dpi=200, bbox_inches='tight')
plt.show()

print("✅ Before/After comparison complete!")
print(f"\nCleaning Impact:")
print(f"   Fixed years: {invalid_train_before}")
print(f"   Year range: {years_before.min():.0f}-{years_before.max():.0f} → {years_after.min():.0f}-{years_after.max():.0f}")
print(f"   Zero popularity handling: Flag feature created")